# TELEPATI 8.0 — AgriData Intelligence Race
## Rice Disease Object Detection — End-to-End Reproducible Pipeline

**Compliance statement**: No external pretrained weights (model built from an
architecture-only `.yaml` definition, `pretrained=False`, verified in Block 6
via source inspection and empty-checkpoint-cache checks). No external
dataset. No LLM/API dataset processing. Official dataset only. Canonical
11-class mapping (verified against the actual dataset with zero unmapped
categories). Deterministic seeding. `YOLO_OFFLINE=1` enforced throughout.

This notebook demonstrates the complete pipeline from raw dataset to final
trained model, reusing the same tested code (`src/agridata/`, `scripts/`)
used and verified throughout development. Full audit trail for every step
below is under `artifacts/` and `artifacts/audit/`.

By default (`SKIP_TRAINING = True`, see Section 12), this notebook loads the
already-trained final model rather than retraining from scratch, so
"Restart & Run All" completes in a few minutes. The exact training code and
frozen configuration used to produce that model are fully shown and
functional — set `SKIP_TRAINING = False` to reproduce the full ~3.3-hour
training run.

## 1. Project & Environment Information

In [ ]:
import sys
from pathlib import Path

# No personal absolute path is hardcoded: resolve the project root relative
# to this notebook's own location, so it works regardless of where the
# repository is cloned.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "agridata").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # allow running from notebooks/ or the project root
assert (PROJECT_ROOT / "src" / "agridata").exists(), "Could not locate the agridata package — run this notebook from the project root or notebooks/."

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from agridata.reproducibility.environment import capture_environment_snapshot

env = capture_environment_snapshot()
print(f"Python version: {env['python_version']}")
print(f"Platform: {env['platform']['platform_string']}")
print(f"Resolved device: {env['device']['resolved_device']} (Apple Silicon: {env['device']['is_apple_silicon']})")
print(f"torch: {env['device']['torch_version']}  |  CUDA available: {env['device']['cuda_available']}  |  MPS available: {env['device']['mps_available']}")
print(f"Git commit (at notebook execution time): {env['git_commit']}")
print(f"Git working tree clean: {env['git_status']['clean']}")

## 2. Imports

In [ ]:
import json
import subprocess
import hashlib
import random

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import yaml

from agridata.seed import set_global_seed
from agridata.device import detect_device
from agridata.dataset.mapping import CANONICAL_CLASSES, build_mapping_report
from agridata.dataset.stats import load_canonical_split
from agridata.visualization.images import draw_annotated_image
from agridata.training.train import build_compliant_model, run_training

## 3. Global Configuration & Dataset Path

The official dataset path is configurable and resolved relative to the
project root (never a personal absolute path). If your dataset lives
elsewhere, change `DATASET_ROOT` below.

In [ ]:
with open(PROJECT_ROOT / "configs" / "final_model_config.yaml") as f:
    FINAL_CONFIG = yaml.safe_load(f)

DATASET_ROOT = PROJECT_ROOT / "Telepati 8.0 Datasets"
assert DATASET_ROOT.exists(), (
    f"Official dataset not found at {DATASET_ROOT}. "
    "Set DATASET_ROOT to wherever you placed the official TELEPATI 8.0 dataset."
)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Final model config ({PROJECT_ROOT / 'configs' / 'final_model_config.yaml'}):")
for k, v in FINAL_CONFIG.items():
    print(f"  {k}: {v}")

## 4. Random Seed

In [ ]:
SEED = FINAL_CONFIG["seed"]
set_global_seed(SEED)
print(f"Global seed set to {SEED} (python random, numpy, torch — see src/agridata/seed.py)")

## 5. Dataset Inspection

Quick summary counts here (fast). The full forensic audit — duplicate
detection, bounding-box validation, cross-split leakage check, image
corruption checks — is a separate, more expensive script:
`scripts/audit_dataset.py` → `artifacts/audit/dataset_audit_report.md`.

In [ ]:
for split in ["train", "valid", "test"]:
    with open(DATASET_ROOT / split / "_annotations.coco.json") as f:
        data = json.load(f)
    print(f"{split:6s}: {len(data['images']):5d} images | {len(data['annotations']):6d} annotations | {len(data['categories'])} raw categories")

## 6. Annotation Inspection & Canonical Class Mapping (11 classes)

In [ ]:
print(f"{len(CANONICAL_CLASSES)} canonical classes:")
for i, c in enumerate(CANONICAL_CLASSES):
    print(f"  model_class_id={i}: {c}")

with open(DATASET_ROOT / "train" / "_annotations.coco.json") as f:
    train_raw = json.load(f)
mapping_report = build_mapping_report(train_raw["categories"])
print(f"\nRaw categories in train split: {mapping_report['total_raw_categories']}")
print(f"Mapped to canonical classes: {len(mapping_report['mapped'])}")
print(f"Supercategory placeholders (excluded, not detection targets): {[p['raw_name'] for p in mapping_report['supercategory_placeholders']]}")
print(f"Unmapped/unknown raw categories: {mapping_report['unmapped_raw_categories']} (must be empty)")
assert not mapping_report["unmapped_raw_categories"], "Unmapped raw category found — see src/agridata/dataset/mapping.py"

## 7. Data Cleaning / Validation — Cross-Split Leakage Check

Loads the existing forensic audit report (see Section 5's note) rather than
re-hashing all ~13,300 images inline, which would make this notebook slow to
run. Regenerate that report with `scripts/audit_dataset.py` if it does not
exist yet.

In [ ]:
audit_report_path = PROJECT_ROOT / "artifacts" / "audit" / "dataset_audit_report.json"
if audit_report_path.exists():
    with open(audit_report_path) as f:
        audit = json.load(f)
    overlap = audit["cross_split_overlap"]["content_hash_overlap"]
    for pair, matches in overlap.items():
        print(f"{pair}: {len(matches)} exact byte-duplicate image(s) found")
        for m in matches:
            print(f"    {m}")
    print("\nConfirmed exact duplicates are excluded from the prepared TRAIN manifest — see Section 8.")
else:
    print(f"No existing audit report at {audit_report_path}. Run: python scripts/audit_dataset.py --dataset-root \"{DATASET_ROOT}\"")

## 8. Dataset Preparation (canonical mapping applied, leakage-excluded, YOLO format)

Builds a training-ready view via symlinks into the raw dataset (no image
bytes are copied). Confirmed exact-duplicate images are excluded from the
TRAIN split only; validation/test point at the original, unmodified
directories.

In [ ]:
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"

if not (PREPARED_DIR / "data.yaml").exists():
    print("Preparing dataset (regenerating the training-ready view)...")
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_dataset.py"),
         "--dataset-root", str(DATASET_ROOT), "--output-dir", str(PREPARED_DIR), "--seed", str(SEED)],
        check=True, cwd=PROJECT_ROOT,
    )
else:
    print(f"Prepared dataset already exists at {PREPARED_DIR}")

with open(PREPARED_DIR / "data.yaml") as f:
    print("\n" + f.read())

## 9. Visualization — Sample Training Images with Ground-Truth Boxes

A small, fixed number of samples (not thousands of embedded images), chosen
deterministically from the global seed.

In [ ]:
split_data = load_canonical_split(DATASET_ROOT, "train", "_annotations.coco.json")
ann_by_image = {}
for ann in split_data.annotations:
    ann_by_image.setdefault(ann.image_id, []).append(ann)
images_by_id = {img.image_id: img for img in split_data.images}

sample_ids = random.sample(sorted(ann_by_image.keys()), 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, image_id in zip(axes, sample_ids):
    img_record = images_by_id[image_id]
    annotated = draw_annotated_image(DATASET_ROOT / "train" / img_record.file_name, img_record, ann_by_image[image_id])
    ax.imshow(annotated)
    ax.set_title(f"image_id={image_id}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 10. Training Configuration (Frozen Final Config)

Selected in Block 14 from 21 controlled screening experiments — see
`artifacts/reports/block14_final_model_selection.md` for the full,
evidence-based reasoning behind every hyperparameter below.

In [ ]:
print(json.dumps(FINAL_CONFIG, indent=2))

## 11. Model Construction (Compliant — No External Pretrained Weights)

`build_compliant_model` refuses to run if `pretrained=True` or if the
architecture argument looks like a checkpoint file (`.pt`/`.pth`/`.ckpt`) —
see `src/agridata/training/train.py`.

In [ ]:
compliant_model = build_compliant_model(FINAL_CONFIG["model_arch"], pretrained=False)
print(f"Model built from architecture-only definition '{FINAL_CONFIG['model_arch']}'. Task: {compliant_model.task}")
print("No external checkpoint was referenced or downloaded (YOLO_OFFLINE enforced).")

## 12. Training (or Loading the Final Model)

`SKIP_TRAINING = True` (default) loads the already-trained final model
(produced by Block 15 using this exact frozen configuration) instead of
retraining, so this notebook completes in a reasonable time when run top to
bottom. Set `SKIP_TRAINING = False` to reproduce the full training run from
scratch (~3.3 hours on this project's Apple Silicon / MPS hardware — see
`artifacts/reports/block15_final_training_summary.json` for the actual
measured duration).

In [ ]:
SKIP_TRAINING = True

FINAL_WEIGHTS_PATH = PROJECT_ROOT / "runs" / "detect" / "final" / "final_model" / "weights" / "best.pt"

if SKIP_TRAINING:
    assert FINAL_WEIGHTS_PATH.exists(), (
        f"SKIP_TRAINING=True but no existing final weights found at {FINAL_WEIGHTS_PATH}. "
        "Set SKIP_TRAINING=False to train from scratch, or run scripts/run_final_training.py first."
    )
    print(f"SKIP_TRAINING=True: loading existing final weights from {FINAL_WEIGHTS_PATH}")
else:
    print("SKIP_TRAINING=False: running full training from the frozen final configuration.")
    print("This reproduces Block 15's training run and will take several hours.")
    extra_kwargs = {
        "optimizer": FINAL_CONFIG["optimizer"], "lr0": FINAL_CONFIG["learning_rate"],
        "momentum": FINAL_CONFIG["momentum"], "weight_decay": FINAL_CONFIG["weight_decay"],
        "patience": FINAL_CONFIG["patience"], "flipud": FINAL_CONFIG.get("flipud", 0.0),
    }
    result = run_training(
        model_arch=FINAL_CONFIG["model_arch"],
        data_yaml=PREPARED_DIR / "data.yaml",
        output_project=PROJECT_ROOT / "runs" / "detect" / "final",
        run_name="final_model_notebook_rerun",
        image_size=FINAL_CONFIG["image_size"],
        batch_size=FINAL_CONFIG["batch_size"],
        epochs=FINAL_CONFIG["epochs"],
        device=detect_device(),
        seed=SEED,
        workers=FINAL_CONFIG["workers"],
        fraction=FINAL_CONFIG["fraction"],
        plots=True,
        validate=True,
        extra_train_kwargs=extra_kwargs,
    )
    FINAL_WEIGHTS_PATH = Path(result["best_weights"])
    print(f"Training complete. Best weights: {FINAL_WEIGHTS_PATH}")

## 13. Validation & Evaluation (mAP@0.5, F1)

Delegates to `scripts/evaluate.py`, run as a subprocess. This is
intentional, not a shortcut: a Block 16 clean-environment reproduction test
found that running Ultralytics' native `val()` and this project's local F1
prediction-collection in the *same* process corrupts MPS backend state on
this hardware; `evaluate.py` runs each stage in its own subprocess to avoid
this reliably. See `artifacts/audit/block16_clean_reproduction_test.md`.

Test-set ground truth is never used here or anywhere in this pipeline for
tuning — only `valid` is evaluated in this notebook.

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate.py"),
     "--weights", str(FINAL_WEIGHTS_PATH),
     "--split", "valid",
     "--prepared-dir", str(PREPARED_DIR),
     "--conf-threshold", "0.25"],
    check=True, cwd=PROJECT_ROOT,
)

with open(PROJECT_ROOT / "artifacts" / "reports" / "evaluation_valid.json") as f:
    eval_report = json.load(f)

## 14. Final Metrics — mAP@0.5 and F1-Score

In [ ]:
native = eval_report["native_metrics"]
local = eval_report["local_f1_metrics"]

print("=== Official metrics (native Ultralytics mAP; local F1 — see src/agridata/metrics/detection.py) ===")
print(f"mAP@0.5:       {native['mAP50']:.4f}")
print(f"mAP@0.5:0.95:  {native['mAP50_95']:.4f}")
print(f"F1 (conf>=0.25): {local['overall']['f1']:.4f}  (precision={local['overall']['precision']:.4f}, recall={local['overall']['recall']:.4f})")
print()
print("Per-class AP@0.5:")
for cls, ap in native["per_class_AP50"].items():
    print(f"  {cls:28s} {ap:.4f}")

## 15. Sample Prediction Visualization

Predicted bounding boxes on real validation images, generated by
`scripts/evaluate.py` above (a small, fixed number of samples).

In [ ]:
pred_samples = sorted((PROJECT_ROOT / "artifacts" / "figures" / "predictions").glob("pred_*.jpg"))[:4]

if pred_samples:
    fig, axes = plt.subplots(1, len(pred_samples), figsize=(20, 5))
    if len(pred_samples) == 1:
        axes = [axes]
    for ax, path in zip(axes, pred_samples):
        ax.imshow(mpimg.imread(path))
        ax.set_title(path.name[:35])
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No prediction samples were saved (no detection exceeded the confidence threshold in the sampled images).")

## 16. Final Model Path & Checksum

In [ ]:
def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print(f"Final model weights: {FINAL_WEIGHTS_PATH}")
print(f"SHA-256 checksum:    {sha256_of_file(FINAL_WEIGHTS_PATH)}")
print(f"File size:           {FINAL_WEIGHTS_PATH.stat().st_size / (1024 * 1024):.2f} MB")

## 17. Standalone Inference Demo

Loads the final weights fresh (independent of any training-time state) and
runs inference on one validation image.

In [ ]:
from ultralytics import YOLO

inference_model = YOLO(str(FINAL_WEIGHTS_PATH))
sample_image_path = sorted((PREPARED_DIR / "valid" / "images").iterdir())[0]
results = inference_model.predict(str(sample_image_path), verbose=False)

print(f"Inference on {sample_image_path.name}: {len(results[0].boxes)} detection(s)")
for box in results[0].boxes:
    cls_name = CANONICAL_CLASSES[int(box.cls.item())]
    conf = float(box.conf.item())
    print(f"  {cls_name}: confidence={conf:.3f}")

## 18. Reproducibility Notes

In [ ]:
with open(PROJECT_ROOT / "artifacts" / "reports" / "block15_final_training_summary.json") as f:
    training_summary = json.load(f)

print("=== Official final training run record (Block 15) ===")
print(f"Git commit:            {training_summary['git_commit']}")
print(f"Dataset manifest hash: {training_summary['dataset_manifest_hash']}")
print(f"Device:                {training_summary['device']}")
print(f"Duration:              {training_summary['duration_seconds'] / 3600:.2f} hours")
print(f"Clean-process load validation: {'PASS' if training_summary['clean_process_load_validation']['success'] else 'FAIL'}")

**Known reproducibility caveats (verified, not hedged speculation):**

- MPS backend (Apple Silicon) has confirmed non-deterministic kernels for
  `scatter_reduce_mps` and `index_put_with_accumulate_mps` (observed as
  explicit PyTorch `UserWarning`s during every training run in this
  project). Exact bit-for-bit training reproducibility is **not**
  guaranteed on this hardware — only reproducible *configuration* and
  *preprocessing* are claimed.
- Dataset preprocessing **is** verified bit-for-bit reproducible: rerunning
  `scripts/prepare_dataset.py` with the same seed produces a byte-for-byte
  identical manifest (Block 8, verified via a live rerun-and-diff).
- One confirmed exact-duplicate image across train/test (Block 2) is
  excluded from the prepared train manifest (Block 5).
- Two real environment/runtime defects were found and fixed via a
  clean-environment reproduction test (Block 16): a `numpy` version pin
  that broke a fresh `pip install`, and an MPS-backend crash on
  large-batch inference (worked around via chunking). See
  `artifacts/audit/block16_clean_reproduction_test.md` for the full
  investigation.
- **Observed, not just theoretical**: rerunning evaluation on the identical
  final checkpoint across separate runs of this notebook produced the exact
  same native mAP@0.5 (0.5620) every time, but the local F1-at-threshold
  metric varied slightly run to run (e.g. 0.3091 vs. 0.3127) — consistent
  with the MPS non-determinism above manifesting at inference time (NMS/
  confidence-score-adjacent ops), not just during training. mAP@0.5 appears
  more robust to this than a single-fixed-threshold F1, likely because it
  integrates over the full precision-recall curve rather than one cutoff.

**Full audit trail**: `artifacts/audit/` (dataset forensic audit,
canonical-mapping validation, reproducibility checklist, clean-environment
reproduction test) and `artifacts/experiments/experiment_log.json` (21
controlled screening experiments underpinning the final configuration
choice, Blocks 10-14).